# Colab: CiteBench Dual Pipeline Evaluation (RAGTruth + LettuceDetect)

This notebook runs both pipelines on CiteBench metric data and saves comparable metrics JSON artifacts to Google Drive.

Outputs are stored under: `/content/drive/MyDrive/AIST-FYP-colab-evals/<timestamp>/`

In [ ]:
# 1) Mount Drive and resolve workspace
from pathlib import Path
import os

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception as exc:
    print(f'Not running in Colab or Drive mount failed: {exc}')

REPO_DIR = Path('/content/AIST-FYP')
if not REPO_DIR.exists():
    print('Repository not found at /content/AIST-FYP. Clone it in the next cell.')
else:
    print(f'Repository found: {REPO_DIR}')

In [ ]:
# 2) Clone repository (if needed) and install dependencies
import subprocess
import shlex

def run_cmd(cmd, cwd=None, check=True):
    if isinstance(cmd, list):
        printable = ' '.join(shlex.quote(str(p)) for p in cmd)
    else:
        printable = cmd
    print(f'>> {printable}')
    proc = subprocess.run(cmd, cwd=str(cwd) if cwd else None, text=True, capture_output=True, shell=isinstance(cmd, str))
    if proc.stdout.strip():
        print(proc.stdout[-2000:])
    if proc.returncode != 0:
        print(proc.stderr[-2000:])
        if check:
            raise RuntimeError(f'Command failed ({proc.returncode}): {printable}')
    return proc

if not REPO_DIR.exists():
    # Update this URL if your fork differs
    run_cmd(['git', 'clone', 'https://github.com/<owner>/AIST-FYP.git', str(REPO_DIR)])

os.chdir(REPO_DIR)

# Preferred: uv project sync for colab env
uv_ok = run_cmd('which uv', check=False).returncode == 0
if not uv_ok:
    run_cmd('curl -LsSf https://astral.sh/uv/install.sh | sh', check=False)
    os.environ['PATH'] = f"{Path.home() / '.local' / 'bin'}:{os.environ.get('PATH','')}"

sync_proc = run_cmd(['uv', 'sync', '--project', 'colab/env', '--extra', 'evaluation'], check=False)
if sync_proc.returncode != 0:
    run_cmd(['pip', 'install', '-r', 'requirements.txt'])

# Ensure runtime dependencies for this workflow
run_cmd(['pip', 'install', '-U', 'lettucedetect', 'python-dotenv'])
print('Dependency setup complete.')

In [ ]:
# 3) Runtime configuration (edit this cell only)
from datetime import datetime
from pathlib import Path

# Data and scale
METRIC_SPLIT = 'test'  # dev or test
MAX_SAMPLES = 10       # smoke default
STRICT = True
INCLUDE_FLAT_CONTEXT = True

SOURCE_METRIC_FILE = REPO_DIR / 'benchmark' / 'CiteEval' / 'data' / 'metric_eval' / f'metric_{METRIC_SPLIT}' / f'citebench.metric_{METRIC_SPLIT}'

# Evaluation provider defaults
CITEEVAL_PROVIDER = 'deepseek'
EVAL_MODEL_NAME = 'deepseek-chat'
MODULES = 'ca,ce,cr_itercoe,cr_editdist'
VERSION = 'citeeval-auto-12272024'
CONTEXT_SOURCE = 'oracle'

# LettuceDetect model
LETTUCE_MODEL_PATH = 'KRLabsOrg/lettucedect-base-modernbert-en-v1'

# RAGTruth mode
# - 'convert_only': convert CiteBench -> RAGTruth-style JSONL, then adapt to CiteEval format
# - 'prediction_input': use an existing RAGTruth prediction file and convert it
RAGTRUTH_MODE = 'convert_only'
RAGTRUTH_PREDICTION_INPUT = ''  # e.g. '/content/drive/MyDrive/path/to/prediction.jsonl'

# Drive output root
DRIVE_ROOT = Path('/content/drive/MyDrive/AIST-FYP-colab-evals')
RUN_STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = DRIVE_ROOT / RUN_STAMP
RAG_DIR = RUN_DIR / 'ragtruth'
LETTUCE_DIR = RUN_DIR / 'lettucedetect'
COMPARE_DIR = RUN_DIR / 'comparison'
EVAL_DIR = RUN_DIR / 'evaluation'

for d in [RUN_DIR, RAG_DIR, LETTUCE_DIR, COMPARE_DIR, EVAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Run directory:', RUN_DIR)
print('Metric source:', SOURCE_METRIC_FILE)
if not SOURCE_METRIC_FILE.exists():
    raise FileNotFoundError(f'Missing CiteBench metric file: {SOURCE_METRIC_FILE}')

In [ ]:
# 4) Load API keys and evaluator env
import json
import os

if IN_COLAB:
    try:
        from google.colab import userdata
        if CITEEVAL_PROVIDER == 'deepseek':
            os.environ['DEEPSEEK_API_KEY'] = userdata.get('DEEPSEEK_API_KEY')
            os.environ.setdefault('DEEPSEEK_BASE_URL', 'https://api.deepseek.com')
        else:
            os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    except Exception as exc:
        print(f'Could not load secret from Colab userdata: {exc}')

os.environ['CITEEVAL_PROVIDER'] = CITEEVAL_PROVIDER
os.environ['CITEEVAL_ROOT'] = str(REPO_DIR / 'benchmark' / 'CiteEval')
extra_pythonpath = os.pathsep.join([
    str(REPO_DIR / 'benchmark' / 'CiteEval'),
    str(REPO_DIR / 'benchmark' / 'CiteEval' / 'src'),
])
os.environ['PYTHONPATH'] = f"{os.environ.get('PYTHONPATH','')}{os.pathsep if os.environ.get('PYTHONPATH') else ''}{extra_pythonpath}"

if CITEEVAL_PROVIDER == 'deepseek' and not os.environ.get('DEEPSEEK_API_KEY'):
    raise RuntimeError('DEEPSEEK_API_KEY is required for DeepSeek evaluation.')
if CITEEVAL_PROVIDER == 'openai' and not os.environ.get('OPENAI_API_KEY'):
    raise RuntimeError('OPENAI_API_KEY is required for OpenAI evaluation.')

print('Evaluator provider:', CITEEVAL_PROVIDER)
print('Environment ready.')

In [ ]:
# 5) Build RAGTruth-side CiteEval input
import json

ragtruth_metric_jsonl = RAG_DIR / f'ragtruth_metric_{METRIC_SPLIT}.jsonl'
ragtruth_aligned_ids = RAG_DIR / 'aligned_ids.json'
ragtruth_convert_report = RAG_DIR / 'convert_metric_report.json'
ragtruth_system_eval = RAG_DIR / 'ragtruth_system_eval.json'
ragtruth_system_report = RAG_DIR / 'ragtruth_to_citeeval_report.json'

if RAGTRUTH_MODE == 'prediction_input':
    if not RAGTRUTH_PREDICTION_INPUT:
        raise ValueError('RAGTRUTH_PREDICTION_INPUT must be set when RAGTRUTH_MODE=prediction_input')
    ragtruth_input_for_adapter = Path(RAGTRUTH_PREDICTION_INPUT)
else:
    run_cmd([
        'python', 'scripts/convert_citebench_metric_to_ragtruth.py',
        '--input', str(SOURCE_METRIC_FILE),
        '--output', str(ragtruth_metric_jsonl),
        '--split', METRIC_SPLIT,
        '--aligned-ids-output', str(ragtruth_aligned_ids),
        '--report-json', str(ragtruth_convert_report),
    ] + (['--max-samples', str(MAX_SAMPLES)] if MAX_SAMPLES is not None else []) + (['--strict'] if STRICT else []), cwd=REPO_DIR)
    ragtruth_input_for_adapter = ragtruth_metric_jsonl

run_cmd([
    'python', 'scripts/convert_ragtruth_baseline_to_citeeval.py',
    '--input', str(ragtruth_input_for_adapter),
    '--output', str(ragtruth_system_eval),
    '--report-json', str(ragtruth_system_report),
] + (['--strict'] if STRICT else []), cwd=REPO_DIR)

print('RAGTruth system-eval file:', ragtruth_system_eval)

In [ ]:
# 6) Run LettuceDetect full pipeline (input conversion + inference + output conversion)
run_cmd([
    'python', 'scripts/run_lettucedetect_pipeline.py',
    '--source-metric-file', str(SOURCE_METRIC_FILE),
    '--metric-split', METRIC_SPLIT,
    '--model-path', LETTUCE_MODEL_PATH,
    '--output-dir', str(LETTUCE_DIR),
] + (['--max-samples', str(MAX_SAMPLES)] if MAX_SAMPLES is not None else [])
  + (['--strict'] if STRICT else [])
  + (['--include-flat-context'] if INCLUDE_FLAT_CONTEXT else []), cwd=REPO_DIR)

lettuce_manifest = LETTUCE_DIR / 'run_manifest.json'
if not lettuce_manifest.exists():
    raise FileNotFoundError(f'Missing LettuceDetect manifest: {lettuce_manifest}')

with open(lettuce_manifest, 'r', encoding='utf-8') as f:
    manifest_obj = json.load(f)

stats = manifest_obj.get('inference_stats', {})
print('LettuceDetect stats:', stats)
if int(stats.get('errors', 0)) > 0:
    raise RuntimeError(f'LettuceDetect inference reported errors: {stats}')

lettuce_system_eval = Path(manifest_obj['system_eval_output'])
print('LettuceDetect system-eval file:', lettuce_system_eval)

In [ ]:
# 7) Compare both methods with identical CiteEval settings
run_cmd([
    'python', 'scripts/compare_citebench_methods.py',
    '--ragtruth-input', str(ragtruth_system_eval),
    '--lettuce-input', str(lettuce_system_eval),
    '--provider', CITEEVAL_PROVIDER,
    '--model-name', EVAL_MODEL_NAME,
    '--modules', MODULES,
    '--version', VERSION,
    '--context-source', CONTEXT_SOURCE,
    '--output-dir', str(COMPARE_DIR),
] + (['--max-samples', str(MAX_SAMPLES)] if MAX_SAMPLES is not None else []), cwd=REPO_DIR)

summary_path = COMPARE_DIR / 'summary.json'
if not summary_path.exists():
    raise FileNotFoundError(f'Missing comparison summary: {summary_path}')

with open(summary_path, 'r', encoding='utf-8') as f:
    comparison_summary = json.load(f)

print('Comparison complete. aligned_count =', comparison_summary.get('run', {}).get('aligned_count'))

In [ ]:
# 8) Save canonical evaluation JSON artifacts on Drive
from datetime import datetime

ragtruth_metrics = {
    'method': 'ragtruth_baseline',
    'split': METRIC_SPLIT,
    'sample_count': comparison_summary.get('run', {}).get('aligned_count'),
    'provider': CITEEVAL_PROVIDER,
    'model_name': EVAL_MODEL_NAME,
    'modules': MODULES,
    'metrics': comparison_summary.get('method_metrics', {}).get('ragtruth', {}),
    'artifact_paths': {
        'system_eval_input': str(ragtruth_system_eval),
        'comparison_summary': str(summary_path),
    },
    'timestamp': datetime.utcnow().isoformat() + 'Z',
}

lettuce_metrics = {
    'method': 'lettucedetect',
    'split': METRIC_SPLIT,
    'sample_count': comparison_summary.get('run', {}).get('aligned_count'),
    'provider': CITEEVAL_PROVIDER,
    'model_name': EVAL_MODEL_NAME,
    'modules': MODULES,
    'metrics': comparison_summary.get('method_metrics', {}).get('lettucedetect', {}),
    'artifact_paths': {
        'system_eval_input': str(lettuce_system_eval),
        'comparison_summary': str(summary_path),
    },
    'timestamp': datetime.utcnow().isoformat() + 'Z',
}

method_comparison = {
    'run': comparison_summary.get('run', {}),
    'commands': comparison_summary.get('commands', {}),
    'delta': comparison_summary.get('delta', {}),
    'timestamp': datetime.utcnow().isoformat() + 'Z',
}

ragtruth_metrics_path = EVAL_DIR / 'ragtruth_metrics.json'
lettuce_metrics_path = EVAL_DIR / 'lettucedetect_metrics.json'
comparison_path = EVAL_DIR / 'method_comparison.json'

with open(ragtruth_metrics_path, 'w', encoding='utf-8') as f:
    json.dump(ragtruth_metrics, f, indent=2, ensure_ascii=False)
with open(lettuce_metrics_path, 'w', encoding='utf-8') as f:
    json.dump(lettuce_metrics, f, indent=2, ensure_ascii=False)
with open(comparison_path, 'w', encoding='utf-8') as f:
    json.dump(method_comparison, f, indent=2, ensure_ascii=False)

run_manifest = {
    'run_dir': str(RUN_DIR),
    'metric_source': str(SOURCE_METRIC_FILE),
    'ragtruth_mode': RAGTRUTH_MODE,
    'provider': CITEEVAL_PROVIDER,
    'model_name': EVAL_MODEL_NAME,
    'max_samples': MAX_SAMPLES,
    'strict': STRICT,
    'paths': {
        'ragtruth_metrics': str(ragtruth_metrics_path),
        'lettucedetect_metrics': str(lettuce_metrics_path),
        'method_comparison': str(comparison_path),
        'comparison_summary': str(summary_path),
    },
}

manifest_path = RUN_DIR / 'run_manifest.json'
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(run_manifest, f, indent=2, ensure_ascii=False)

print('Saved JSON artifacts:')
print('-', ragtruth_metrics_path)
print('-', lettuce_metrics_path)
print('-', comparison_path)
print('-', manifest_path)

## Notes

- `RAGTRUTH_MODE='convert_only'` uses CiteBench predictions as the response field for a baseline-compatible conversion path.
- To evaluate an official RAGTruth baseline prediction file, set `RAGTRUTH_MODE='prediction_input'` and provide `RAGTRUTH_PREDICTION_INPUT`.
- The canonical comparison artifact for downstream verifier alignment is `evaluation/method_comparison.json`.